# Boosting: Correcting Mistakes Sequentially

AdaBoost starts with equal observation weights. After each decision stump, misclassified observations receive more influence. The final prediction is a weighted vote of all fitted stumps.

## Learning goals

- Trace the role of observation weights in AdaBoost.
- Interpret estimator error and estimator voting weight.
- Explain the tradeoff between `learning_rate` and `n_estimators`.

In [ ]:
from pathlib import Path
import sys

import numpy as np

repo_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "src" / "rice_ml").is_dir()
)
sys.path.insert(0, str(repo_root / "src"))

from rice_ml.supervised_ml import AdaBoostClassifier

In [ ]:
rng = np.random.default_rng(8)
X = rng.uniform(-2.5, 2.5, size=(240, 2))
y = np.where(X[:, 0] * X[:, 1] > 0, "same sign", "different sign")

# Flip a few labels to make the task imperfect and expose boosting's sensitivity.
noise = rng.choice(len(y), size=16, replace=False)
y[noise] = np.where(y[noise] == "same sign", "different sign", "same sign")

indices = rng.permutation(len(X))
train, test = indices[:170], indices[170:]
X_train, X_test = X[train], X[test]
y_train, y_test = y[train], y[test]

In [ ]:
boost = AdaBoostClassifier(n_estimators=50, learning_rate=0.6, random_state=8)
boost.fit(X_train, y_train)
predictions = boost.predict(X_test)

print(f"test accuracy: {np.mean(predictions == y_test):.3f}")
print(f"stumps retained: {len(boost.estimators_)}")
print("first five stump errors:", np.round(boost.estimator_errors_[:5], 3))
print("first five voting weights:", np.round(boost.estimator_weights_[:5], 3))

In [ ]:
print(f"{'round':>5} {'weighted error':>16} {'vote weight':>14}")
for round_number, (error, alpha) in enumerate(
    zip(boost.estimator_errors_, boost.estimator_weights_), start=1
):
    print(f"{round_number:5d} {error:16.3f} {alpha:14.3f}")

A stump with lower weighted error receives a larger vote. Later errors are measured on a changed distribution, so they are not directly comparable to ordinary test error.

## Experiment

Try learning rates 0.1, 0.5, 1.0, and 1.5. For each value, record test accuracy and the number of retained stumps. Then increase the number of flipped labels and describe when boosting begins to chase noise.